# Northstar Retail - individual portal build
Student notebook | 2026-2027

Follow TP_Day_04.pdf. Required: create your streaming environment, run cells 0-9 individually, complete section 10 and clean up. Optional A/B extends the same environment with batch/ADF and Load Balancer. All processing code is supplied. Four non-secret configuration edits only. Do not use Run all.

## 0 - Configuration
Replace CATALOG, SCHEMA, EH_NAMESPACE, SERVICE_CREDENTIAL. Create the personal schema and managed stream_state volume in the UI first.

In [ ]:
import re, json
from pathlib import Path
from decimal import Decimal
from pyspark.sql import functions as F

CATALOG = "REPLACE_CATALOG"
SCHEMA = "northstar4_REPLACE_LEARNER"
EH_NAMESPACE = "REPLACE_NAMESPACE.servicebus.windows.net"
EH_NAME = "sales-live"
SERVICE_CREDENTIAL = "REPLACE_SERVICE_CREDENTIAL"
for name in (CATALOG, SCHEMA, SERVICE_CREDENTIAL):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name)
    assert "REPLACE" not in name, "Replace the four configuration strings using your created resources"
assert SCHEMA.startswith("northstar4_")
assert CATALOG.lower() not in {"system", "samples", "hive_metastore"}
assert re.fullmatch(r"[a-z0-9-]+\.servicebus\.windows\.net", EH_NAMESPACE)
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", EH_NAME)
NS = f"`{CATALOG}`.`{SCHEMA}`"
def table(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{NS}.`{name}`"
BATCH_BRONZE = table("sales_batch_bronze")
BATCH_SILVER = table("sales_batch_silver")
BATCH_GOLD = table("sales_batch_gold")
STREAM_BRONZE = table("sales_stream_bronze")
STREAM_GOLD = table("sales_stream_gold")
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Create schema and volume in Catalog Explorer before running this cell.
STATE = f"/Volumes/{CATALOG}/{SCHEMA}/stream_state"
BRONZE_CP = f"{STATE}/sales_live/bronze_v1"
GOLD_CP = f"{STATE}/sales_live/gold_v1"
assert BRONZE_CP != GOLD_CP
spark.sql("SELECT CURRENT_METASTORE() AS metastore_id").show(truncate=False)
dbutils.fs.ls(STATE)
EXPECTED = json.loads('{"currency": "EUR", "source_rows": 32, "adf_rows_read": 32, "adf_rows_copied": 32, "batch_bronze_rows": 32, "batch_silver_rows": 24, "batch_gold_rows": 4, "batch_total_quantity": 48, "batch_total_revenue": "411.40", "batch_by_store": [{"StoreID": "S01", "StoreName": "Paris Centre", "Region": "Ile-de-France", "TransactionCount": 6, "TotalQuantity": 12, "TotalRevenue": "67.50"}, {"StoreID": "S02", "StoreName": "Versailles", "Region": "Ile-de-France", "TransactionCount": 6, "TotalQuantity": 12, "TotalRevenue": "138.20"}, {"StoreID": "S03", "StoreName": "Lyon Centre", "Region": "Auvergne-Rhone-Alpes", "TransactionCount": 6, "TotalQuantity": 12, "TotalRevenue": "67.50"}, {"StoreID": "S04", "StoreName": "Grenoble", "Region": "Auvergne-Rhone-Alpes", "TransactionCount": 6, "TotalQuantity": 12, "TotalRevenue": "138.20"}], "stream": {"event_count": 24, "bronze_rows": 24, "unique_sale_ids": 24, "gold_rows": 12, "total_quantity": 48, "total_revenue": "411.40", "windows": [{"store_id": "S01", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "12.00"}, {"store_id": "S01", "window_start": "2026-10-01T08:01:00Z", "window_end": "2026-10-01T08:02:00Z", "transaction_count": 2, "total_quantity": 4, "total_revenue": "26.25"}, {"store_id": "S01", "window_start": "2026-10-01T08:02:00Z", "window_end": "2026-10-01T08:03:00Z", "transaction_count": 2, "total_quantity": 5, "total_revenue": "29.25"}, {"store_id": "S02", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 5, "total_revenue": "49.20"}, {"store_id": "S02", "window_start": "2026-10-01T08:01:00Z", "window_end": "2026-10-01T08:02:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "43.90"}, {"store_id": "S02", "window_start": "2026-10-01T08:02:00Z", "window_end": "2026-10-01T08:03:00Z", "transaction_count": 2, "total_quantity": 4, "total_revenue": "45.10"}, {"store_id": "S03", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 4, "total_revenue": "26.25"}, {"store_id": "S03", "window_start": "2026-10-01T08:01:00Z", "window_end": "2026-10-01T08:02:00Z", "transaction_count": 2, "total_quantity": 5, "total_revenue": "29.25"}, {"store_id": "S03", "window_start": "2026-10-01T08:02:00Z", "window_end": "2026-10-01T08:03:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "12.00"}, {"store_id": "S04", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "43.90"}, {"store_id": "S04", "window_start": "2026-10-01T08:01:00Z", "window_end": "2026-10-01T08:02:00Z", "transaction_count": 2, "total_quantity": 4, "total_revenue": "45.10"}, {"store_id": "S04", "window_start": "2026-10-01T08:02:00Z", "window_end": "2026-10-01T08:03:00Z", "transaction_count": 2, "total_quantity": 5, "total_revenue": "49.20"}]}, "demo_stream": {"event_count": 8, "bronze_rows": 8, "unique_sale_ids": 8, "gold_rows": 4, "total_quantity": 15, "total_revenue": "131.35", "windows": [{"store_id": "S01", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "12.00"}, {"store_id": "S02", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 5, "total_revenue": "49.20"}, {"store_id": "S03", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 4, "total_revenue": "26.25"}, {"store_id": "S04", "window_start": "2026-10-01T08:00:00Z", "window_end": "2026-10-01T08:01:00Z", "transaction_count": 2, "total_quantity": 3, "total_revenue": "43.90"}]}, "source_sha256": {"sales_batch_final.csv": "23ed6db90740d3237c70f40716aa480e41b21a8d572d56f760c7f0448b09b905", "products.csv": "0798649c3f5fe2a19d1c996c6b54c1b3779132b7a918030e5ac2fd85e8699cc1", "stores.csv": "25b688da14aad8b9b26c6387b20466a6cf82254ee2c645f8eca7c2de81e89c71"}}')

print("PASS configuration: existing personal schema/volume, UTC, embedded unchanged expectations")
print(STREAM_BRONZE, STREAM_GOLD, BRONZE_CP, GOLD_CP, sep="\n")


## 1 - Inspect initial state
Run after configuration; first-run streaming tables must not already exist. This cell does not delete anything.

In [ ]:
for name in (STREAM_BRONZE, STREAM_GOLD):
    print("Existing table:", name, spark.catalog.tableExists(name))
print("Volume entries:", [x.name for x in dbutils.fs.ls(STATE)])
print("FIRST RUN: require no streaming tables and an empty personal volume. For recovery, retain matching existing state.")


## 2 - Producer installation command
Run to PRINT the complete command. Copy only the printed script to your Azure VM: Operations > Run command > RunShellScript. It installs software inside your VM; it does not create Azure resources and does not send events.

In [ ]:
import base64
INSTALL_TEMPLATE = 'set -eu\nexport DEBIAN_FRONTEND=noninteractive\napt-get update\napt-get install -y python3-venv python3-pip ca-certificates\ninstall -d -m 0755 /opt/northstar-producer\npython3 - <<\'PYFILES\'\nimport base64\nfrom pathlib import Path\np=Path(\'/opt/northstar-producer\')\n(p/\'producer.py\').write_bytes(base64.b64decode(\'IiIiU2VuZCBkZXRlcm1pbmlzdGljIE5vcnRoc3RhciBzYWxlcyBmcm9tIGFuIEF6dXJlIFZNIHVzaW5nIGl0cyBtYW5hZ2VkIGlkZW50aXR5LiIiIg0KaW1wb3J0IGFyZ3BhcnNlDQpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZWRlbHRhLCB0aW1lem9uZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBtYXRoDQppbXBvcnQgb3MNCmltcG9ydCByZQ0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCg0KUFJJQ0VTX0NFTlRTID0gKDQ1MCwgMTIwMCwgNzI1LCAxOTkwLCAzNzUsIDg0MCkNCkJBU0VfRVZFTlRfVElNRSA9IGRhdGV0aW1lKDIwMjYsIDEwLCAxLCA4LCAwLCB0emluZm89dGltZXpvbmUudXRjKQ0KDQoNCmRlZiBnZW5lcmF0ZV9ldmVudHMoY291bnQ9MjQsIHN0YXJ0X2luZGV4PTEsIHByZWZpeD0iTElWRSIpOg0KICAgIGlmIG5vdCAxIDw9IGNvdW50IDw9IDEwMDAgb3Igbm90IDEgPD0gc3RhcnRfaW5kZXggPD0gOTk5MDAwOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb3VudCBtdXN0IGJlIDEuLjEwMDAgYW5kIHN0YXJ0LWluZGV4IDEuLjk5OTAwMCIpDQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyIltBLVpdW0EtWjAtOV17MCwxMX0iLCBwcmVmaXgpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcmVmaXggbXVzdCBiZSAxLi4xMiB1cHBlcmNhc2UgbGV0dGVycy9kaWdpdHMiKQ0KICAgIGZvciBudW1iZXIgaW4gcmFuZ2Uoc3RhcnRfaW5kZXgsIHN0YXJ0X2luZGV4ICsgY291bnQpOg0KICAgICAgICBpID0gbnVtYmVyIC0gMQ0KICAgICAgICBzdGFtcCA9IEJBU0VfRVZFTlRfVElNRSArIHRpbWVkZWx0YShzZWNvbmRzPShpIC8vIDgpKjYwICsgKGkgJSA4KSo1KQ0KICAgICAgICB5aWVsZCB7DQogICAgICAgICAgICAic2FsZV9pZCI6IGYie3ByZWZpeH0te251bWJlcjowNmR9IiwNCiAgICAgICAgICAgICJzdG9yZV9pZCI6IGYiU3soaSAlIDQpKzE6MDJkfSIsDQogICAgICAgICAgICAicHJvZHVjdF9pZCI6IGYiUHsoaSAlIDYpKzE6MDJkfSIsDQogICAgICAgICAgICAicXVhbnRpdHkiOiAoaSAlIDMpKzEsDQogICAgICAgICAgICAidW5pdF9wcmljZSI6IFBSSUNFU19DRU5UU1tpICUgNl0gLyAxMDAsDQogICAgICAgICAgICAiZXZlbnRfdGltZSI6IHN0YW1wLmlzb2Zvcm1hdCgpLnJlcGxhY2UoIiswMDowMCIsICJaIiksDQogICAgICAgIH0NCg0KDQpkZWYgc2VuZF9ldmVudHMoZXZlbnRzLCBuYW1lc3BhY2UsIGh1YiwgYXV0aD0iZW50cmEiLCBkZWxheT0wLjEpOg0KICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbYS16MC05LV0rXC5zZXJ2aWNlYnVzXC53aW5kb3dzXC5uZXQiLCBuYW1lc3BhY2UpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTZXQgRVZFTlRIVUJfTkFNRVNQQUNFIHRvIG5hbWUuc2VydmljZWJ1cy53aW5kb3dzLm5ldCIpDQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyIltBLVphLXowLTldW0EtWmEtejAtOS5fLV17MCwyNTR9IiwgaHViKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2V0IEVWRU5USFVCX05BTUUgdG8geW91ciBkZWRpY2F0ZWQgZXZlbnQgaHViIG5hbWUiKQ0KICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKGRlbGF5KSBvciBub3QgMCA8PSBkZWxheSA8PSAxMDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGVsYXkgbXVzdCBiZSBhIGZpbml0ZSB2YWx1ZSBiZXR3ZWVuIDAgYW5kIDEwIikNCiAgICB0cnk6DQogICAgICAgIGZyb20gYXp1cmUuZXZlbnRodWIgaW1wb3J0IEV2ZW50RGF0YSwgRXZlbnRIdWJQcm9kdWNlckNsaWVudA0KICAgICAgICBmcm9tIGF6dXJlLmlkZW50aXR5IGltcG9ydCBEZWZhdWx0QXp1cmVDcmVkZW50aWFsDQogICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGV4YzoNCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJJbnN0YWxsIHBhY2thZ2VzIGZyb20gcmVxdWlyZW1lbnRzLnR4dCIpIGZyb20gZXhjDQogICAgaWYgYXV0aCAhPSAiZW50cmEiOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbmx5IEF6dXJlIFZNIG1hbmFnZWQtaWRlbnRpdHkgYXV0aGVudGljYXRpb24gaXMgc3VwcG9ydGVkIikNCiAgICBjcmVkZW50aWFsID0gRGVmYXVsdEF6dXJlQ3JlZGVudGlhbCgNCiAgICAgICAgZXhjbHVkZV9lbnZpcm9ubWVudF9jcmVkZW50aWFsPVRydWUsDQogICAgICAgIGV4Y2x1ZGVfd29ya2xvYWRfaWRlbnRpdHlfY3JlZGVudGlhbD1UcnVlLA0KICAgICAgICBleGNsdWRlX3NoYXJlZF90b2tlbl9jYWNoZV9jcmVkZW50aWFsPVRydWUsDQogICAgICAgIGV4Y2x1ZGVfdmlzdWFsX3N0dWRpb19jb2RlX2NyZWRlbnRpYWw9VHJ1ZSwNCiAgICAgICAgZXhjbHVkZV9jbGlfY3JlZGVudGlhbD1UcnVlLA0KICAgICAgICBleGNsdWRlX3Bvd2Vyc2hlbGxfY3JlZGVudGlhbD1UcnVlLA0KICAgICAgICBleGNsdWRlX2RldmVsb3Blcl9jbGlfY3JlZGVudGlhbD1UcnVlLA0KICAgICAgICBleGNsdWRlX2ludGVyYWN0aXZlX2Jyb3dzZXJfY3JlZGVudGlhbD1UcnVlLA0KICAgICAgICBleGNsdWRlX2Jyb2tlcl9jcmVkZW50aWFsPVRydWUpDQogICAgY2xpZW50ID0gRXZlbnRIdWJQcm9kdWNlckNsaWVudCgNCiAgICAgICAgZnVsbHlfcXVhbGlmaWVkX25hbWVzcGFjZT1uYW1lc3BhY2UsIGV2ZW50aHViX25hbWU9aHViLA0KICAgICAgICBjcmVkZW50aWFsPWNyZWRlbnRpYWwsIHJldHJ5X3RvdGFsPTIpDQogICAgc2VudCA9IDANCiAgICB0cnk6DQogICAgICAgIHdpdGggY2xpZW50Og0KICAgICAgICAgICAgZm9yIGV2ZW50IGluIGV2ZW50czoNCiAgICAgICAgICAgICAgICBwYXlsb2FkID0ganNvbi5kdW1wcyhldmVudCwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpLCBlbnN1cmVfYXNjaWk9RmFsc2UpDQogICAgICAgICAgICAgICAgYmF0Y2ggPSBjbGllbnQuY3JlYXRlX2JhdGNoKHBhcnRpdGlvbl9rZXk9ZXZlbnRbInN0b3JlX2lkIl0pDQogICAgICAgICAgICAgICAgYmF0Y2guYWRkKEV2ZW50RGF0YShwYXlsb2FkKSkNCiAgICAgICAgICAgICAgICBjbGllbnQuc2VuZF9iYXRjaChiYXRjaCwgdGltZW91dD0zMCkNCiAgICAgICAgICAgICAgICBzZW50ICs9IDENCiAgICAgICAgICAgICAgICBwcmludCgiU0VOVCAiICsgcGF5bG9hZCwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgICAgICBpZiBkZWxheToNCiAgICAgICAgICAgICAgICAgICAgdGltZS5zbGVlcChkZWxheSkNCiAgICBmaW5hbGx5Og0KICAgICAgICBpZiBjcmVkZW50aWFsIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgY3JlZGVudGlhbC5jbG9zZSgpDQogICAgcmV0dXJuIHNlbnQNCg0KDQpkZWYgbWFpbihhcmd2PU5vbmUpOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb3VudCIsIHR5cGU9aW50LCBkZWZhdWx0PTI0KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc3RhcnQtaW5kZXgiLCB0eXBlPWludCwgZGVmYXVsdD0xKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHJlZml4IiwgZGVmYXVsdD0iTElWRSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZWxheSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4xLA0KICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iUGF1c2UgYmV0d2VlbiBzZW5kcywgaW4gc2Vjb25kcyIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PSIvb3B0L25vcnRoc3Rhci1wcm9kdWNlci9jb25maWcuanNvbiIsDQogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJWTS1vd25lZCBKU09OIHdpdGggbmFtZXNwYWNlIGFuZCBodWI7IG5vIHNlY3JldHMiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZHJ5LXJ1biIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsDQogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJQcmludCBKU09OIHdpdGhvdXQgaW1wb3J0aW5nIFNES3Mgb3IgY29udGFjdGluZyBBenVyZSIpDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpDQogICAgdHJ5Og0KICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShhcmdzLmRlbGF5KSBvciBub3QgMCA8PSBhcmdzLmRlbGF5IDw9IDEwOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGVsYXkgbXVzdCBiZSBhIGZpbml0ZSB2YWx1ZSBiZXR3ZWVuIDAgYW5kIDEwIikNCiAgICAgICAgZXZlbnRzID0gbGlzdChnZW5lcmF0ZV9ldmVudHMoYXJncy5jb3VudCwgYXJncy5zdGFydF9pbmRleCwgYXJncy5wcmVmaXgpKQ0KICAgICAgICBpZiBhcmdzLmRyeV9ydW46DQogICAgICAgICAgICBmb3IgZXZlbnQgaW4gZXZlbnRzOg0KICAgICAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMoZXZlbnQsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkpDQogICAgICAgICAgICBwcmludChmIkRSWSBSVU46IHtsZW4oZXZlbnRzKX0gZXZlbnRzOyBub3RoaW5nIHNlbnQiLCBmaWxlPXN5cy5zdGRlcnIpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCiAgICAgICAgICAgIGNvbmZpZ19wYXRoID0gUGF0aChhcmdzLmNvbmZpZykNCiAgICAgICAgICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoY29uZmlnX3BhdGgucmVhZF90ZXh0KCkpIGlmIGNvbmZpZ19wYXRoLmV4aXN0cygpIGVsc2Uge30NCiAgICAgICAgICAgIG5hbWVzcGFjZSA9IG9zLmdldGVudigiRVZFTlRIVUJfTkFNRVNQQUNFIikgb3IgY29uZmlnLmdldCgibmFtZXNwYWNlIiwgIiIpDQogICAgICAgICAgICBodWIgPSBvcy5nZXRlbnYoIkVWRU5USFVCX05BTUUiKSBvciBjb25maWcuZ2V0KCJodWIiLCAiIikNCiAgICAgICAgICAgIHNlbnQgPSBzZW5kX2V2ZW50cyhldmVudHMsIG5hbWVzcGFjZSwgaHViLCBkZWxheT1hcmdzLmRlbGF5KQ0KICAgICAgICAgICAgcHJpbnQoZiJDT01QTEVURToge3NlbnR9IGV2ZW50cyBzZW50IikNCiAgICAgICAgcmV0dXJuIDANCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFJ1bnRpbWVFcnJvcikgYXMgZXhjOg0KICAgICAgICBwcmludChmIkNPTkZJR1VSQVRJT04gRVJST1I6IHtleGN9IiwgZmlsZT1zeXMuc3RkZXJyKQ0KICAgICAgICByZXR1cm4gMg0KICAgIGV4Y2VwdCBLZXlib2FyZEludGVycnVwdDoNCiAgICAgICAgcHJpbnQoIlN0b3BwZWQuIFNvbWUgZXZlbnRzIG1heSBhbHJlYWR5IGhhdmUgYmVlbiBzZW50OyBpbnNwZWN0IGJlZm9yZSByZXRyeWluZy4iLCBmaWxlPXN5cy5zdGRlcnIpDQogICAgICAgIHJldHVybiAxMzANCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgIyBTREsgZXhjZXB0aW9uIGRldGFpbHMgY2FuIGluY2x1ZGUgc2Vuc2l0aXZlIGNvbm5lY3Rpb24gY29uZmlndXJhdGlvbi4NCiAgICAgICAgcHJpbnQoZiJTRU5EIEZBSUxFRCAoe3R5cGUoZXhjKS5fX25hbWVfX30pLiBDaGVjayBWTSBtYW5hZ2VkIGlkZW50aXR5LCBTZW5kIHJvbGUsICINCiAgICAgICAgICAgICAgIm5hbWVzcGFjZSwgaHViIGFuZCBuZXR3b3JrLiBTb21lIHNlbmRzIG1heSBoYXZlIHN1Y2NlZWRlZC4gIg0KICAgICAgICAgICAgICAiRG8gbm90IHJlcnVuIGJsaW5kbHk7IGR1cGxpY2F0ZSBidXNpbmVzcyBldmVudHMgYXJlIHBvc3NpYmxlLiIsIGZpbGU9c3lzLnN0ZGVycikNCiAgICAgICAgcmV0dXJuIDENCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K\'))\n(p/\'lab.json\').write_bytes(base64.b64decode(\'CONFIG64\'))\nPYFILES\ncd /opt/northstar-producer\npython3 -m venv .venv\n.venv/bin/python -m pip install \'azure-eventhub>=5.12,<6\' \'azure-identity>=1.17,<2\'\n.venv/bin/python -c "import azure.eventhub, azure.identity; print(\'SDKs ready\')"\necho \'INSTALLATION READY - nothing sent\'\n'
config64 = base64.b64encode(json.dumps({"namespace": EH_NAMESPACE, "hub": EH_NAME}).encode()).decode()
print(INSTALL_TEMPLATE.replace("CONFIG64", config64))


## 3 - Event Hubs source
First finish the producer dry-run and ONE real send of 24 LIVE events in the PDF. Run supplied source code unchanged.

In [ ]:
kafka_options = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}:9093",
    "subscribe": EH_NAME,
    "databricks.serviceCredential": SERVICE_CREDENTIAL,
    "startingOffsets": "earliest",
    "failOnDataLoss": "true",
    "maxOffsetsPerTrigger": "1000",
}
# Spark manages a query-specific Kafka group. Never share an explicit group.id.
kafka_stream = spark.readStream.format("kafka").options(**kafka_options).load()
assert kafka_stream.isStreaming
kafka_stream.printSchema()


## 4 - Parse the JSON
Run unchanged; inspect the streaming schema. No write query has started yet.

In [ ]:
event_schema = """
    sale_id STRING, store_id STRING, product_id STRING,
    quantity INT, unit_price DECIMAL(10,2), event_time TIMESTAMP
"""
incoming = (kafka_stream
    .select(F.col("value").cast("string").alias("raw_json"),
            "topic", "partition", "offset",
            F.col("timestamp").alias("broker_time"))
    .withColumn("event", F.from_json("raw_json", event_schema,
                                    {"mode": "PERMISSIVE"}))
    .select("raw_json", "topic", "partition", "offset", "broker_time", "event.*")
    .withColumn("processing_time", F.current_timestamp()))
incoming.printSchema()
# Keep malformed events in Bronze. Filter analytical fields downstream.


## 5 - Query helper
Run unchanged. This helper stops only the query passed to it if it does not finish.

In [ ]:
def finish_query(query, timeout_seconds=180):
    try:
        if not query.awaitTermination(timeout_seconds):
            raise TimeoutError("Query still active; inspect progress and resume")
        if query.exception() is not None:
            raise RuntimeError(str(query.exception()))
        print("Query finished:", query.name, "active:", query.isActive)
        print("Last progress:", query.lastProgress)
    finally:
        if query.isActive:
            query.stop()
# Run the finite producer BEFORE starting AvailableNow on a fresh hub.


## 6 - Streaming Bronze
Run after producer COMPLETE. Expect 24 records and inactive query.

In [ ]:
bronze_query = (incoming.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CP)
    .queryName("northstar4_bronze")
    .trigger(availableNow=True)
    .toTable(STREAM_BRONZE))
finish_query(bronze_query)
spark.table(STREAM_BRONZE).select(
    "sale_id", "store_id", "quantity", "unit_price", "event_time",
    "broker_time", "processing_time", "partition", "offset").orderBy("sale_id").show(30, False)
assert spark.table(STREAM_BRONZE).count() == EXPECTED["stream"]["bronze_rows"]
print("Checkpoint entries:", len(dbutils.fs.ls(BRONZE_CP)))
print("PASS Bronze:", spark.table(STREAM_BRONZE).count())


## 7 - Streaming Gold
Run supplied aggregation and write unchanged. Expect 12 rows,24 transactions,48 units, EUR 411.40.

In [ ]:
bronze_stream = spark.readStream.table(STREAM_BRONZE)
valid_events = bronze_stream.filter(
    F.col("sale_id").isNotNull() & (F.trim("sale_id") != "") &
    F.col("store_id").isNotNull() & (F.trim("store_id") != "") &
    F.col("product_id").isNotNull() & F.col("event_time").isNotNull() &
    (F.col("quantity") > 0) & (F.col("unit_price") > 0))
with_revenue = valid_events.withColumn(
    "revenue", (F.col("quantity") * F.col("unit_price")).cast("decimal(18,2)"))
windows = with_revenue.groupBy("store_id", F.window("event_time", "1 minute")).agg(
    F.count("*").alias("transaction_count"),
    F.sum("quantity").alias("total_quantity"),
    F.sum("revenue").cast("decimal(20,2)").alias("total_revenue"))
stream_gold = windows.select(
    "store_id", F.col("window.start").alias("window_start"),
    F.col("window.end").alias("window_end"),
    "transaction_count", "total_quantity", "total_revenue")
gold_query = (stream_gold.writeStream.format("delta")
    .outputMode("complete")
    .option("checkpointLocation", GOLD_CP)
    .queryName("northstar4_gold")
    .trigger(availableNow=True)
    .toTable(STREAM_GOLD))
finish_query(gold_query)
spark.table(STREAM_GOLD).orderBy("window_start", "store_id").show(30, False)
assert not bronze_query.isActive and not gold_query.isActive
assert spark.table(STREAM_GOLD).count() == 12
spark.table(STREAM_GOLD).agg(F.sum("transaction_count"), F.sum("total_quantity"), F.sum("total_revenue")).show()
print("PASS Gold: 12 store/window rows")


## 8 - Checkpoint recovery
DO NOT rerun producer. Run this cell using the same tables and the same two checkpoints.

In [ ]:
before_bronze = spark.table(STREAM_BRONZE).count()
before_gold = spark.table(STREAM_GOLD).orderBy("store_id", "window_start").collect()
# Reuse source, destinations and checkpoint paths. Do NOT rerun producer.
replay_bronze = (incoming.writeStream.format("delta").outputMode("append")
    .option("checkpointLocation", BRONZE_CP)
    .trigger(availableNow=True).toTable(STREAM_BRONZE))
finish_query(replay_bronze)
replay_gold = (stream_gold.writeStream.format("delta").outputMode("complete")
    .option("checkpointLocation", GOLD_CP)
    .trigger(availableNow=True).toTable(STREAM_GOLD))
finish_query(replay_gold)
assert spark.table(STREAM_BRONZE).count() == before_bronze
assert spark.table(STREAM_GOLD).orderBy("store_id", "window_start").collect() == before_gold
print("PASS checkpoint recovery: no new input, no changed result")
assert before_bronze == 24 and len(before_gold) == 12
assert not replay_bronze.isActive and not replay_gold.isActive
print("Before / after Bronze:", before_bronze, spark.table(STREAM_BRONZE).count())
print("Before / after Gold:", len(before_gold), spark.table(STREAM_GOLD).count())


## 9 - Required final validation
Read-only checks for streaming only. Batch and Load Balancer are optional and are not needed for this PASS.

In [ ]:
"""Read-only checks for the dedicated Northstar Day 4 tables."""
import re
from decimal import Decimal

def validate_stream(spark, catalog, schema, expected):
    from pyspark.sql import functions as F
    for value in (catalog, schema):
        assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value)
    assert schema.startswith("northstar4_")
    ns = f"`{catalog}`.`{schema}`"
    def read(name):
        qualified = f"{ns}.`{name}`"
        assert spark.sql(f"DESCRIBE DETAIL {qualified}").first()["format"] == "delta"
        return spark.table(qualified)
    sb = read("sales_stream_bronze")
    sg = read("sales_stream_gold")
    e = expected["stream"]
    assert sb.count() == sb.select("sale_id").distinct().count() == e["bronze_rows"]
    assert sb.select("topic", "partition", "offset").distinct().count() == e["bronze_rows"]
    assert {r.sale_id for r in sb.select("sale_id").collect()} == {f"LIVE-{i:06}" for i in range(1,25)}
    types = dict(sb.dtypes)
    for name, typ in {"sale_id":"string", "quantity":"int", "unit_price":"decimal(10,2)", "event_time":"timestamp", "processing_time":"timestamp"}.items():
        assert types[name] == typ, (name, types[name])
    assert sb.filter(F.col("event_time").isNull() | F.col("raw_json").isNull() | F.col("processing_time").isNull()).count() == 0
    assert sg.count() == e["gold_rows"]
    assert sg.select("store_id", "window_start").distinct().count() == e["gold_rows"]
    def iso(t): return t.strftime("%Y-%m-%dT%H:%M:%SZ")
    actual_windows = sorted((r.store_id, iso(r.window_start), iso(r.window_end), r.transaction_count, r.total_quantity, str(r.total_revenue)) for r in sg.collect())
    target_windows = sorted((r["store_id"], r["window_start"], r["window_end"], r["transaction_count"], r["total_quantity"], r["total_revenue"]) for r in e["windows"])
    assert actual_windows == target_windows, (actual_windows, target_windows)
    totals = sg.agg(F.sum("transaction_count"), F.sum("total_quantity"), F.sum("total_revenue")).first()
    assert tuple(totals) == (e["event_count"], e["total_quantity"], Decimal(e["total_revenue"]))
    print("PASS required streaming: Bronze24, Gold12, transactions24, units48, EUR411.40")
    return {"stream": "PASS"}
try:
    validate_stream(spark, CATALOG, SCHEMA, EXPECTED)
except AssertionError as exc:
    raise AssertionError("Streaming check failed. Read the failing assertion and compare the PDF expected results. Do not resend or erase checkpoints.") from exc
assert not bronze_query.isActive and not gold_query.isActive
assert not replay_bronze.isActive and not replay_gold.isActive


## 10 - Required evidence and short answers
Double-click this Markdown cell to edit, then Shift+Enter to render. Keep actual resource values and results here; no separate report.

| Evidence | Your observation |
|---|---|
| Personal resource group; region; workspace URL; catalog/schema | TODO |
| VNet/subnet; VM private IP; public IP for outbound; inbound NSG policy | TODO |
| VM managed identity / Data Sender scope | TODO |
| Access connector identity / Data Receiver scope / service credential | TODO |
| Hub name; tier; partition count; retention | TODO |
| Installation READY; dry-run first/last IDs; real send COMPLETE once | TODO |
| Bronze count; Gold count; totals; inactive queries | TODO |
| Recovery before/after counts and unchanged results | TODO |

1. For one sale_id, record event_time, broker_time, processing_time, partition and offset. What does each mean? TODO
2. Why did recovery preserve the results? Would sending the same sale_id again have the same effect? TODO
3. Which resources store data, execute processing and govern access? TODO

Required route complete after validation and these answers. Follow PDF cleanup unless you choose an optional extension. Optional sections below are NOT required for the streaming PASS.


## Optional A0 - Actual ADF landing path
Run only after completing the PDF optional storage/ADF/volume setup. This reads the real ADF landing output.

In [ ]:
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing_files"
REF = Path(LANDING) / "reference"
BATCH_FILE = f"{LANDING}/batch/sales_batch_final.csv"
for path in (BATCH_FILE, str(REF/"products.csv"), str(REF/"stores.csv"), str(REF/"expected_results.json")):
    dbutils.fs.ls(path)
assert json.loads((REF/"expected_results.json").read_text(encoding="utf-8-sig")) == EXPECTED
print("PASS optional ADF path and unchanged expected-results file")


## Optional A1 - Batch Bronze
Run unchanged after optional A0. Preserve original batch expected results: 32 -> 24 ->4.

In [ ]:
batch_schema = """
    SaleID STRING, SaleDate STRING, StoreID STRING,
    ProductID STRING, Quantity STRING, UnitPrice STRING
"""
raw = (spark.read.option("header", True).option("mode", "FAILFAST")
    .schema(batch_schema).csv(BATCH_FILE))
assert raw.count() == EXPECTED["batch_bronze_rows"]
raw.write.format("delta").mode("overwrite").saveAsTable(BATCH_BRONZE)
print("PASS batch Bronze:", raw.count())


## Optional A2 - Batch Silver
Run unchanged after optional A0. Preserve original batch expected results: 32 -> 24 ->4.

In [ ]:
raw = spark.table(BATCH_BRONZE)
# Exact source duplicates only; conflicting records require investigation.
typed = (raw.dropDuplicates()
    .withColumn("SaleDate", F.expr("try_cast(SaleDate AS DATE)"))
    .withColumn("Quantity", F.expr("try_cast(Quantity AS INT)"))
    .withColumn("UnitPrice", F.expr("try_cast(UnitPrice AS DECIMAL(10,2))")))
silver = typed.filter(
    F.col("SaleID").isNotNull() & (F.trim("SaleID") != "") &
    F.col("SaleDate").isNotNull() &
    (F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
assert silver.select("SaleID").distinct().count() == silver.count()
silver.write.format("delta").mode("overwrite").saveAsTable(BATCH_SILVER)
assert silver.count() == EXPECTED["batch_silver_rows"]
print("PASS batch Silver:", silver.count())


## Optional A3 - Batch Gold
Run unchanged after optional A0. Preserve original batch expected results: 32 -> 24 ->4.

In [ ]:
products = (spark.read.option("header", True)
    .schema("ProductID STRING, ProductName STRING, Category STRING")
    .csv(str(REF / "products.csv")))
stores = (spark.read.option("header", True)
    .schema("StoreID STRING, StoreName STRING, Region STRING")
    .csv(str(REF / "stores.csv")))
assert products.count() == products.select("ProductID").distinct().count() == 6
assert stores.count() == stores.select("StoreID").distinct().count() == 4
silver = spark.table(BATCH_SILVER)
assert silver.join(products, "ProductID", "left_anti").count() == 0
assert silver.join(stores, "StoreID", "left_anti").count() == 0
enriched = (silver.join(products, "ProductID").join(stores, "StoreID")
    .withColumn("Revenue", (F.col("Quantity")*F.col("UnitPrice"))
                .cast("decimal(18,2)")))
batch_gold = enriched.groupBy("StoreID", "StoreName", "Region").agg(
    F.count("*").alias("TransactionCount"),
    F.sum("Quantity").alias("TotalQuantity"),
    F.sum("Revenue").cast("decimal(20,2)").alias("TotalRevenue"))
batch_gold.write.format("delta").mode("overwrite").saveAsTable(BATCH_GOLD)
spark.table(BATCH_GOLD).orderBy("StoreID").show(truncate=False)


## Optional A4 - Full five-table validation
Run only after the optional batch cells. The two EUR 411.40 workloads are separate; do not add them.

In [ ]:
"""Read-only checks for the dedicated Northstar Day 4 tables."""
import re
from decimal import Decimal

def validate_day04(spark, catalog, schema, expected):
    from pyspark.sql import functions as F
    for value in (catalog, schema):
        assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value)
    assert schema.startswith("northstar4_")
    ns = f"`{catalog}`.`{schema}`"
    def read(name):
        qualified = f"{ns}.`{name}`"
        assert spark.sql(f"DESCRIBE DETAIL {qualified}").first()["format"] == "delta"
        return spark.table(qualified)
    b = read("sales_batch_bronze")
    s = read("sales_batch_silver")
    g = read("sales_batch_gold")
    assert b.count() == expected["batch_bronze_rows"]
    assert s.count() == s.select("SaleID").distinct().count() == expected["batch_silver_rows"]
    assert dict(s.dtypes)["Quantity"] == "int"
    assert dict(s.dtypes)["UnitPrice"] == "decimal(10,2)"
    assert dict(s.dtypes)["SaleDate"] == "date"
    assert g.count() == g.select("StoreID").distinct().count() == expected["batch_gold_rows"]
    actual = {r.StoreID: (r.TransactionCount, r.TotalQuantity, str(r.TotalRevenue)) for r in g.collect()}
    target = {r["StoreID"]: (r["TransactionCount"], r["TotalQuantity"], r["TotalRevenue"]) for r in expected["batch_by_store"]}
    assert actual == target, (actual, target)
    sb = read("sales_stream_bronze")
    sg = read("sales_stream_gold")
    e = expected["stream"]
    assert sb.count() == sb.select("sale_id").distinct().count() == e["bronze_rows"]
    assert sb.select("topic", "partition", "offset").distinct().count() == e["bronze_rows"]
    assert {r.sale_id for r in sb.select("sale_id").collect()} == {f"LIVE-{i:06}" for i in range(1,25)}
    types = dict(sb.dtypes)
    for name, typ in {"sale_id":"string", "quantity":"int", "unit_price":"decimal(10,2)", "event_time":"timestamp", "processing_time":"timestamp"}.items():
        assert types[name] == typ, (name, types[name])
    assert sb.filter(F.col("event_time").isNull() | F.col("raw_json").isNull() | F.col("processing_time").isNull()).count() == 0
    assert sg.count() == e["gold_rows"]
    assert sg.select("store_id", "window_start").distinct().count() == e["gold_rows"]
    def iso(t): return t.strftime("%Y-%m-%dT%H:%M:%SZ")
    actual_windows = sorted((r.store_id, iso(r.window_start), iso(r.window_end), r.transaction_count, r.total_quantity, str(r.total_revenue)) for r in sg.collect())
    target_windows = sorted((r["store_id"], r["window_start"], r["window_end"], r["transaction_count"], r["total_quantity"], r["total_revenue"]) for r in e["windows"])
    assert actual_windows == target_windows, (actual_windows, target_windows)
    totals = sg.agg(F.sum("transaction_count"), F.sum("total_quantity"), F.sum("total_revenue")).first()
    assert tuple(totals) == (e["event_count"], e["total_quantity"], Decimal(e["total_revenue"]))
    print("PASS: five Delta tables; batch and streaming counts, grain, types and all Gold values")
    print("Also record the actual ADF run and Azure network observations from the Lab.")
    return {"batch": "PASS", "stream": "PASS", "cloud_evidence": "record separately"}
validate_day04(spark, CATALOG, SCHEMA, EXPECTED)


## Optional A/B - Architecture and network answers
Record actual ADF run ID; fileName; source and sink paths; rowsRead 32 / rowsCopied 32; status. TODO
If you complete the LB extension, record both backend private IPs, NSG HTTP/probe rules, pool, probe, frontend response and independent health for each backend. TODO

1. Why is the Load Balancer unrelated to ADF orchestration? TODO
2. Why use ADF for a bounded file and Event Hubs for events? TODO
3. Why can Databricks process both? TODO
4. What does Delta provide? TODO
5. How does fileName differ from a checkpoint? TODO
6. Which components store data, execute compute and govern access? TODO
7. Which resource owns the frontend public IP and why can backends remain private? TODO
8. Explain a health-probe failure caused by the service, port/path or NSG. TODO
